- **Module:** read_and_download_aeronet.ipynb
- **Authors:** Petar Grigorov
- **Organization:** NASA AERONET (https://aeronet.gsfc.nasa.gov/)
- **Date:** 06/18/2023
- **Last Revision:** 05/27/2025
- **Purpose:** To access and download AERONET data from Web API
- **Disclaimer:** The code is for demonstration purposes only. Users are responsible to check for accuracy and revise to fit their objective.

**Required packages installation, library importing and warning handling**

In [ ]:
from bs4 import BeautifulSoup      #reads data from website (web scraping)
import requests                    #useful for sending HTTP requests using python
import numpy as np                 #for array manipulation
import datetime                    #for time data manipulation
import pandas as pd                #for data querying and processing

from google.colab import files      #ensures output zip file can be downloaded
from google.colab import drive      #imports local google drive
drive.mount('/drive')               #mounts local google drive onto colab
!mkdir AERONET                      #makes directory where output file will be stored

**Main Program**

In [ ]:
url = input("Specify URL: ")
average_type = int(input("Specify Average Type (1 for daily, 2 for hourly, 3 for all points): "))
soup = BeautifulSoup(requests.get(url).text)

with open(r'/content/temp.txt' ,"w") as oFile:
    oFile.write(str(soup.text))
    oFile.close()

with open(r'/content/temp.txt', "r") as oFile:
    lines = oFile.readlines()

header_row = None

for i, line in enumerate(lines):
    if "AERONET_Site" in line:
        header_row = i
        break

if len(lines) > 2:
  df = pd.read_csv(r'/content/temp.txt',skiprows=header_row)
  !rm temp.txt

  if not df.empty:
    df.replace(-999.0, np.nan, inplace=True)
    date_split = df['Date(dd:mm:yyyy)'].str.split(':', expand=True)
    df['Date'] = pd.to_datetime(dict(year=date_split[2], month=date_split[1], day=date_split[0]))
    df['Hour'] = df['Time(hh:mm:ss)'].str[:2]
    numeric_cols = df.select_dtypes(include=['number']).columns

    if average_type == 1:
      df = df.groupby(['AERONET_Site', 'Date'])[numeric_cols].mean()
      df = df.reset_index()
    elif average_type == 2:
      df = df.groupby(['AERONET_Site', 'Date','Hour'])[numeric_cols].mean()
      df = df.reset_index()
    elif average_type == 3:
      df.set_index(df.columns[:3].tolist(), inplace=True)
      df = df.select_dtypes(include=['number'])
      df = df.reset_index()
    else:
      average_type = 'daily'
      df = df.groupby(['AERONET_Site', 'Date'])[numeric_cols].mean()
      df = df.reset_index()
      print("\nIncorrect input for average type. Defaulting to daily averages.")

    df.to_csv('/content/AERONET/AERONET_output.csv', index=False)
    files.download('/content/AERONET/AERONET_output.csv')
  else:
    print("Dataframe is empty. Please retry with different parameters.")

else:
  !rm temp.txt
  print("No data to parse. Please retry with different parameters.")